# 🌌 GalaxyChop — Recorrido por todas sus funcionalidades

Este notebook muestra, paso a paso, **todo lo que se puede hacer con GalaxyChop**:
leer y guardar galaxias, inspeccionarlas, calcular energías, preprocesarlas,
estudiar su dinámica estelar, descomponerlas en componentes, graficarlas,
armar pipelines y usar la línea de comandos.

La idea es que cada autor lo corra de punta a punta y vea cada funcionalidad en acción.

## Contenido

1. Carga de la galaxia de ejemplo
2. El objeto `Galaxy`
3. Los conjuntos de partículas (`ParticleSet`)
4. Construir una galaxia desde arreglos (`mkgalaxy`)
5. Entrada / Salida (HDF5 y NumPy)
6. Energía potencial
7. Preprocesamiento: centrado, alineación y recorte
8. Dinámica estelar (circularidad)
9. Descomposición dinámica (6 métodos)
10. Galaxias descompuestas (`DecomposedGalaxy`)
11. Gráficos
12. Pipelines
13. Línea de comandos
14. Configuración y constantes

**Requisitos:** GalaxyChop instalado y el notebook ejecutado desde la raíz del repositorio
(usa la galaxia de ejemplo `tests/datasets/gal394242.h5`).

---
## 1. Carga de la galaxia de ejemplo

La galaxia de ejemplo pertenece a la simulación EAGLE (ID 394242) y ya incluye el
potencial gravitatorio de cada partícula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

import galaxychop as gchop

gchop.__version__

In [ ]:
gal = gchop.read_hdf5("tests/datasets/gal394242.h5")
gal

---
## 2. El objeto `Galaxy`

Una `Galaxy` agrupa tres conjuntos de partículas: estrellas, materia oscura y gas.
Es **inmutable**: todas las operaciones devuelven una galaxia nueva.

In [ ]:
print(repr(gal))
print("Total de partículas:", len(gal))
print("¿Tiene potencial?:", gal.has_potential_)

Cantidad de partículas y masa total por tipo:

In [ ]:
gal.total_mass()

### 2.1 Propiedades físicas derivadas

Cada propiedad devuelve una tupla `(estrellas, materia oscura, gas)` con unidades de astropy.

In [ ]:
k_s, k_dm, k_g = gal.kinetic_energy_       # energía cinética específica
p_s, p_dm, p_g = gal.potential_energy_     # energía potencial específica
e_s, e_dm, e_g = gal.total_energy_         # energía total específica
J_s, J_dm, J_g = gal.angular_momentum_     # momento angular específico (3, n)
r_s, r_dm, r_g = gal.radius_               # distancia al origen
vc_s, vc_dm, vc_g = gal.galaxy_circular_velocity_  # v_circ con la masa de toda la galaxia

print("Energía cinética estrellas:", k_s[:3])
print("Energía total estrellas   :", e_s[:3])
print("Momento angular estrellas :", J_s.shape, J_s.unit)
print("Radio estrellas           :", r_s[:3])
print("V. circular (galaxia)     :", vc_s[:3])

### 2.2 Conversión a tablas y diccionarios

In [ ]:
# Todas las partículas con todos sus atributos
gal.to_dataframe()

In [ ]:
# Sólo algunos tipos de partícula y algunos atributos
gal.to_dataframe(ptypes=["stars", "gas"], attributes=["ptype", "m", "x", "y", "z", "Jz"])

In [ ]:
# Diccionario por tipo de partícula
d = gal.to_dict(attributes=["m", "radius"])
d.keys(), d["stars"].keys()

In [ ]:
# Copia de la galaxia
gal_copia = gal.copy()
gal_copia

---
## 3. Los conjuntos de partículas (`ParticleSet`)

Cada tipo de partícula es un `ParticleSet` con masas, posiciones, velocidades,
potencial y softening, más los atributos derivados (energías, momento angular, radio,
velocidad circular).

In [ ]:
stars = gal.stars
stars

In [ ]:
print(repr(stars))
print("Tipo           :", stars.ptype, stars.ptype.emoji, stars.ptype.humanize())
print("Masa total     :", stars.total_mass())
print("Masas          :", stars.m[:3])
print("Posiciones x   :", stars.x[:3])
print("Velocidades vx :", stars.vx[:3])
print("Potencial      :", stars.potential[:3])
print("Softening      :", stars.softening)
print("Jz             :", stars.Jz_[:3])
print("E. cinética    :", stars.kinetic_energy_[:3])
print("E. total       :", stars.total_energy_[:3])
print("Radio          :", stars.radius_[:3])
print("V. circular (sólo estrellas):", stars.ptype_circular_velocity_[:3])

El accesor `arr_` devuelve los mismos valores como arreglos de NumPy sin unidades:

In [ ]:
stars.arr_.x[:3], stars.arr_.Jz_[:3]

In [ ]:
stars.to_dataframe(attributes=["m", "x", "y", "z", "total_energy", "Jz"])

Los tipos de partícula se representan con `ParticleSetType` y se pueden obtener
desde el nombre o el número:

In [ ]:
gchop.ParticleSetType.mktype("dark_matter"), gchop.ParticleSetType.mktype(2), list(gchop.ParticleSetType)

---
## 4. Construir una galaxia desde arreglos (`mkgalaxy`)

`mkgalaxy` crea una galaxia a partir de arreglos con sufijos `_s` (estrellas),
`_dm` (materia oscura) y `_g` (gas). Las unidades por defecto son
$M_\odot$, kpc y km/s; el potencial y el softening son opcionales.

`disassemble()` hace la operación inversa: convierte una galaxia en ese diccionario.

In [ ]:
partes = gal.disassemble()
sorted(partes)

In [ ]:
gal_reconstruida = gchop.mkgalaxy(**partes)
gal_reconstruida

Ejemplo: una galaxia más chica tomando una de cada 30 partículas, y sin potencial.

In [ ]:
partes_chicas = {k: (v if np.ndim(v) == 0 else v[::30]) for k, v in partes.items()}
partes_chicas.update(potential_s=None, potential_dm=None, potential_g=None)

gal_chica = gchop.mkgalaxy(**partes_chicas)
gal_chica

Los atributos aceptan cantidades de astropy en cualquier unidad equivalente:

In [ ]:
pset = gchop.ParticleSet(
    gchop.ParticleSetType.STARS,
    m=[1e6, 2e6] * u.Msun,
    x=[100, 200] * u.pc,      # se interpretan correctamente en kpc
    y=[0, 0], z=[0, 0],
    vx=[0, 0], vy=[200, 150], vz=[0, 0],
    potential=None,
    softening=0.1,
)
print(pset.x, "→", pset.arr_.x, "kpc")
print("Jz:", pset.Jz_)

---
## 5. Entrada / Salida

### 5.1 HDF5

GalaxyChop guarda y lee galaxias (y galaxias descompuestas) en HDF5. Se pueden
agregar metadatos propios y guardar varias galaxias en un mismo archivo usando grupos.

In [ ]:
gchop.to_hdf5("ejemplo_galaxia.h5", gal, metadata={"simulacion": "EAGLE", "id": 394242}, force_group=True)
gchop.read_hdf5("ejemplo_galaxia.h5")

In [ ]:
# Atajo desde el objeto, guardando otra galaxia en un grupo distinto del mismo archivo
gal_chica.to_hdf5("ejemplo_galaxia.h5", group="chica", force_group=True)
gchop.read_hdf5("ejemplo_galaxia.h5", group="chica")

In [ ]:
# El softening de cada tipo de partícula se define al leer
gal_soft = gchop.read_hdf5("tests/datasets/gal394242.h5", softening_s=0.1, softening_dm=0.5, softening_g=0.1)
gal_soft.stars.softening, gal_soft.dark_matter.softening, gal_soft.gas.softening

### 5.2 Archivos NumPy (formato legacy)

`read_npy` lee un `.npy` por tipo de partícula, indicando el nombre de cada columna,
y opcionalmente los potenciales en archivos separados.

In [ ]:
gal_npy = gchop.io.read_npy(
    "tests/datasets/star_ID_394242.npy",
    "tests/datasets/dark_ID_394242.npy",
    "tests/datasets/gas_ID_394242.npy",
    columns=["m", "x", "y", "z", "vx", "vy", "vz", "id"],
    path_or_stream_pot_s="tests/datasets/potential_star_ID_394242.npy",
    path_or_stream_pot_dm="tests/datasets/potential_dark_ID_394242.npy",
    path_or_stream_pot_g="tests/datasets/potential_gas_ID_394242.npy",
)
gal_npy

---
## 6. Energía potencial

Si la galaxia no trae potencial, GalaxyChop lo calcula por suma directa con tres
*backends*: `numba` (por defecto, paralelo), `numpy` y `grispy`.

Usamos la galaxia chica (sin potencial) para que sea rápido.

In [ ]:
gal_chica.has_potential_

In [ ]:
gal_chica_pot = gchop.preproc.potential(gal_chica)  # backend numba
gal_chica_pot

In [ ]:
gal_chica_pot.stars.potential[:5]

In [ ]:
# Otros backends
gchop.preproc.potential(gal_chica, backend="numpy").stars.potential[:5]

In [ ]:
gchop.preproc.potential(gal_chica, backend="grispy").stars.potential[:5]

La misma operación como objeto transformador (útil en pipelines):

In [ ]:
potentializer = gchop.preproc.Potentializer(backend="numba")
print(potentializer.checker(gal_chica), "→", potentializer.checker(potentializer.transform(gal_chica)))

---
## 7. Preprocesamiento

Antes de descomponer, la galaxia debe estar **centrada** y **alineada** (momento angular
estelar en la dirección $+z$). Cada paso existe como función y como transformador.

### 7.1 Centrado

In [ ]:
gchop.preproc.is_centered(gal)

In [ ]:
# Centra en la partícula de menor potencial y resta la velocidad del CM estelar
gal_c = gchop.preproc.center(gal)
gchop.preproc.is_centered(gal_c)

In [ ]:
# Alternativa: centrar en el centro de masa estelar (no requiere potencial)
gal_c_cm = gchop.preproc.center(gal, with_potential=False)
gal_c_cm

### 7.2 Alineación

In [ ]:
gal_ca = gchop.preproc.star_align(gal_c)
gchop.preproc.is_star_aligned(gal_c), gchop.preproc.is_star_aligned(gal_ca)

In [ ]:
# Usando sólo las estrellas dentro de 30 kpc para definir la orientación
gal_ca30 = gchop.preproc.star_align(gal_c, r_cut=30)
gal_ca30

### 7.3 Centrado y alineación en un solo paso

In [ ]:
gal_ca = gchop.preproc.center_and_align(gal)
gchop.preproc.is_centered_and_aligned(gal_ca)

### 7.4 Recorte por radio de media masa estelar

Elimina las estrellas más allá de `num_radii` veces el radio que encierra la mitad de
la masa estelar (por defecto 3).

In [ ]:
from galaxychop.preproc.smr_crop import get_radius_half_mass

print("Radio de media masa estelar:", get_radius_half_mass(gal_ca), "kpc")
gal_cut = gchop.preproc.half_star_mass_radius_crop(gal_ca, num_radii=3)
gal_cut

### 7.5 Transformadores

In [ ]:
centralizer = gchop.preproc.Centralizer()
aligner = gchop.preproc.Aligner(r_cut=30)
cutter = gchop.preproc.Cutter(num_radii=3)

g = cutter.transform(aligner.transform(centralizer.transform(gal)))
print(centralizer, aligner, cutter)
g

---
## 8. Dinámica estelar

`stellar_dynamics()` calcula, para cada estrella:

* `normalized_star_energy`: energía específica normalizada,
* `normalized_star_Jz`: momento angular en $z$ normalizado,
* `eps`: parámetro de circularidad $\epsilon = J_z/J_{circ}(E)$,
* `eps_r`: circularidad proyectada $J_p/J_{circ}(E)$,

y la curva `(x, y)` usada para obtener $J_{circ}(E)$. Es la base de todos los descomponedores.

In [ ]:
gsd = gal_ca.stellar_dynamics()
gsd

In [ ]:
gsd.to_dataframe()

In [ ]:
print("Atributos:", gsd.circularity_attributes())
print("Estrellas con valores finitos:", gsd.isfinite().sum(), "de", len(gal_ca.stars))
print("Curva J_circ:", len(gsd.x), "puntos")

Parámetros: tamaño de los bins de energía (`bin0`, `bin1`) y qué hacer con las estrellas
con $|\epsilon| > 1$ (`reassign=True` las lleva a ±1, `False` las descarta).

In [ ]:
gsd_r = gal_ca.stellar_dynamics(bin0=0.05, bin1=0.005, reassign=True)
gsd_r.isfinite().sum()

---
## 9. Descomposición dinámica

GalaxyChop incluye seis métodos. Todos se usan igual: se crea el descomponedor con sus
hiperparámetros y se llama a `decompose(galaxia)`.

| Método | Espacio | Componentes |
|---|---|---|
| `JThreshold` | $\epsilon$ | esferoide / disco por un corte en $\epsilon$ |
| `JHistogram` | $\epsilon$ | esferoide / disco (Abadi et al. 2003) |
| `JEHistogram` | $\epsilon$, $E$ | esferoide / disco balanceando por energía |
| `KMeans` | $E$, $\epsilon$, $\epsilon_r$ | `n_components` grupos |
| `GaussianMixture` | $E$, $\epsilon$, $\epsilon_r$ | `n_components` gaussianas, con probabilidades |
| `AutoGaussianMixture` | $E$, $\epsilon$, $\epsilon_r$ | halo, bulbo, disco frío y disco tibio, con probabilidades |

In [ ]:
gchop.decomposers.__all__

### 9.1 `JThreshold`

In [ ]:
jthreshold = gchop.decomposers.JThreshold(eps_cut=0.6)
jthreshold

In [ ]:
dg_jt = jthreshold.decompose(gal_ca)
dg_jt

### 9.2 `JHistogram`

In [ ]:
dg_jh = gchop.decomposers.JHistogram(n_bin=100, random_state=42).decompose(gal_ca)
dg_jh

### 9.3 `JEHistogram`

In [ ]:
dg_jeh = gchop.decomposers.JEHistogram(n_bin=100, n_bin_E=20, random_state=42).decompose(gal_ca)
dg_jeh

### 9.4 `KMeans`

In [ ]:
dg_km = gchop.decomposers.KMeans(n_components=3, random_state=42).decompose(gal_ca)
dg_km

### 9.5 `GaussianMixture`

In [ ]:
dg_gmm = gchop.decomposers.GaussianMixture(n_components=3, random_state=42).decompose(gal_ca)
dg_gmm

### 9.6 `AutoGaussianMixture`

Elige automáticamente la cantidad de gaussianas (puede tardar alrededor de un minuto;
`n_jobs=-1` usa todos los núcleos).

In [ ]:
dg_auto = gchop.decomposers.AutoGaussianMixture(c_bic=0.1, random_state=42, n_jobs=-1).decompose(gal_ca)
dg_auto

### 9.7 Hiperparámetros comunes

Todos los descomponedores aceptan `cbins` (bins de la dinámica estelar) y `reassign`.
Los hiperparámetros se pasan siempre por nombre.

In [ ]:
gchop.decomposers.JThreshold(eps_cut=0.5, cbins=(0.1, 0.01), reassign=True).decompose(gal_ca)

> **Nuevos métodos:** cualquier descomponedor nuevo se puede escribir heredando de
> `gchop.decomposers.GalaxyDecomposerABC` e implementando `get_attributes()` y `split()`
> (y opcionalmente `get_component_name_mapping()`); los hiperparámetros se declaran con
> `gchop.decomposers.hparam(default=...)`.

---
## 10. Galaxias descompuestas (`DecomposedGalaxy`)

El resultado de una descomposición es una galaxia que además sabe a qué componente
pertenece cada partícula. Tiene todo lo de una `Galaxy` más la información de componentes.

In [ ]:
print(repr(dg_auto))
print("Método           :", dg_auto.method)
print("Componentes      :", dg_auto.component_name_mapping)
print("Probabilidades   :", dg_auto.has_probabilities)
print("Etiquetas únicas :", dg_auto.unique_components_labels)

Masa por componente: cantidad de partículas, masa (`m`), fracción de la masa estelar
(`mf`) y, en los métodos probabilísticos, masa ponderada por probabilidad (`pm`, `pmf`).

In [ ]:
dg_auto.total_mass()

In [ ]:
dg_auto.stars

In [ ]:
s = dg_auto.stars
print("Componente de cada estrella:", s.components[:5])
print("Etiqueta de cada estrella  :", s.labels[:5])
print("Probabilidades             :", s.probabilities.shape)
print("V. circular por componente :", s.component_circular_velocity_[:5])

In [ ]:
dg_auto.to_dataframe(ptypes=["stars"], attributes=["m", "x", "y", "z", "components", "labels", "prob_0", "prob_1", "prob_2", "prob_3"])

In [ ]:
# Volver a una Galaxy común
dg_auto.to_galaxy()

In [ ]:
# Las galaxias descompuestas también se guardan y leen en HDF5
dg_auto.to_hdf5("ejemplo_descompuesta.h5", force_group=True)
gchop.read_hdf5("ejemplo_descompuesta.h5")

---
## 11. Gráficos

Toda galaxia tiene el accesor `plot` con los gráficos:
`hist`, `hist2d`, `kde`, `kde2d`, `rotation_curve`, `sdyn_hist`, `sdyn_hist2d`,
`sdyn_kde`, `sdyn_kde2d`. Todos devuelven un `Axes` de matplotlib y aceptan `ax=`.

### 11.1 Galaxia

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
gal_ca.plot.hist2d(x="x", y="y", ptypes=["stars"], bins=200, ax=axs[0])
axs[0].set_title("Estrellas de cara")
gal_ca.plot.hist2d(x="x", y="z", ptypes=["stars"], bins=200, ax=axs[1])
axs[1].set_title("Estrellas de canto")
gal_ca.plot.hist2d(x="x", y="y", bins=200, ax=axs[2])
axs[2].set_title("Todas las partículas");

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
gal_ca.plot.hist("Jz", ptypes=["stars", "gas"], stat="density", common_norm=False, element="step", ax=axs[0])
gal_ca.plot.rotation_curve(ax=axs[1])
axs[1].set_title("Curva de rotación");

Los KDE son más costosos, así que usamos la galaxia chica:

In [ ]:
gal_chica_ca = gchop.preproc.center_and_align(gal_chica_pot)

fig, axs = plt.subplots(1, 2, figsize=(14, 6))
gal_chica_ca.plot.kde2d(x="x", y="z", ptypes=["stars", "gas"], ax=axs[0])
gal_chica_ca.plot.kde("radius", ax=axs[1]);

### 11.2 Dinámica estelar

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 6))
gal_ca.plot.sdyn_hist2d(bins=100, ax=axs[0])
gal_ca.plot.sdyn_hist("eps", bins=100, ax=axs[1])
gal_chica_ca.plot.sdyn_kde2d(x="normalized_star_energy", y="eps", ax=axs[2]);

### 11.3 Galaxias descompuestas

Los mismos gráficos, coloreados por componente.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(18, 12))
dg_auto.plot.sdyn_hist2d(bins=100, ax=axs[0, 0])
axs[0, 0].set_title("AutoGaussianMixture")
dg_jh.plot.sdyn_hist2d(bins=100, ax=axs[0, 1])
axs[0, 1].set_title("JHistogram")
dg_km.plot.sdyn_hist2d(bins=100, ax=axs[0, 2])
axs[0, 2].set_title("KMeans")
dg_auto.plot.hist2d(x="x", y="z", ptypes=["stars"], bins=150, ax=axs[1, 0])
dg_auto.plot.sdyn_hist("eps", bins=100, element="step", ax=axs[1, 1])
dg_auto.plot.rotation_curve(ptypes=["stars"], ax=axs[1, 2])
plt.tight_layout()

### 11.4 Opciones de los gráficos

* Se puede llamar `galaxia.plot("nombre_del_grafico", ...)`.
* `lmap` renombra las etiquetas de la leyenda (diccionario o función).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 6))
gal_ca.plot("hist2d", x="x", y="z", bins=200, ax=axs[0],
            lmap={"stars": "Estrellas", "dark_matter": "Materia oscura", "gas": "Gas"})
dg_jh.plot.hist2d(x="x", y="z", ptypes=["stars"], bins=150, ax=axs[1], lmap=str.upper);

---
## 12. Pipelines

Un pipeline encadena transformadores y termina en un descomponedor. `mkpipe`
les pone nombre automáticamente.

In [ ]:
pipe = gchop.mkpipe(
    gchop.preproc.Centralizer(),
    gchop.preproc.Aligner(r_cut=30),
    gchop.preproc.Cutter(num_radii=3),
    gchop.decomposers.JHistogram(random_state=42),
)
pipe

In [ ]:
# Aplica todo el preprocesamiento y la descomposición sobre la galaxia original
pipe.decompose(gal)

In [ ]:
# Sólo el preprocesamiento
pipe.transform(gal)

In [ ]:
# Acceso a los pasos
print(len(pipe))
print(pipe.steps)
print(pipe[0])
print(pipe["cutter"])
print(pipe.named_steps.aligner)
print(pipe[1:])  # sub-pipeline

In [ ]:
# Pipeline con nombres explícitos
gchop.GchopPipeline([
    ("centrado", gchop.preproc.Centralizer()),
    ("alineado", gchop.preproc.Aligner()),
    ("metodo", gchop.decomposers.KMeans(n_components=2, random_state=42)),
])

---
## 13. Línea de comandos

Al instalar GalaxyChop queda disponible el comando `gchop`.

In [ ]:
!gchop --help

Métodos de descomposición disponibles:

In [ ]:
!gchop methods

Resumen de una galaxia guardada en HDF5:

In [ ]:
!gchop info tests/datasets/gal394242.h5

Centrar, alinear, descomponer y guardar el resultado en un solo comando:

In [ ]:
!gchop decompose --help

In [ ]:
!gchop decompose tests/datasets/gal394242.h5 ejemplo_cli.h5 --method JHistogram --overwrite

Hacer un gráfico y guardarlo en un archivo:

In [ ]:
!gchop plot ejemplo_cli.h5 --kind sdyn_hist2d -o ejemplo_cli.png

In [ ]:
from IPython.display import Image
Image("ejemplo_cli.png")

---
## 14. Configuración y constantes

Estilos de los gráficos por componente (colores, líneas, orden de dibujo) y constantes
físicas usadas por el paquete.

In [ ]:
print("G =", gchop.constants.G, gchop.constants.G_UNIT)
print("Bins por defecto de la dinámica estelar:", gchop.constants.SD_DEFAULT_CBIN)
print("Orden de dibujo:", gchop.constants.PLOT_ORDER)

In [ ]:
for nombre, estilo in gchop.config.items():
    print(f"{nombre:13s}", estilo.get_mplstyle())

---
## Limpieza

Borra los archivos generados durante el recorrido.

In [ ]:
import pathlib

for archivo in ["ejemplo_galaxia.h5", "ejemplo_descompuesta.h5", "ejemplo_cli.h5", "ejemplo_cli.png"]:
    pathlib.Path(archivo).unlink(missing_ok=True)